# Analysis of the ACER Paper model

With this notebook you can reproduce Figures 9 to 18 of the ACER paper "Preparing for the European flexibility needs assessment" (ACER, 2026), the *ACER Paper*, from your own run of its test model, and use it as a starting point for plots of your own. The ACER Paper model is a stylised system of three bidding zones (`N1`, `N2`, `N3`) for target year 2030 and climate year 2013, over a full year of hourly snapshots. It is stored in `data/teaser/`; see the User Manual (`doc/user_manual.md`), section 2.1, for what it contains. Your results can differ slightly from those in the paper, because the solver does not always return exactly the same solution.

The notebook reads the outputs of the first run of the ACER Paper model, `runs/teaser` (configuration `config/teaser.yaml`). Section 2.8 of the User Manual explains step by step how to run it, install Jupyter and run this notebook.

| Figure | Plot | Read from |
|:---|:---|:---|
| 9 | Annual generation per carrier and bidding zone, against the demand | day-ahead network |
| 10 | Renewable generation, curtailment and load spillage per bidding zone | intraday network |
| 11 | Residual load: day-ahead forecast against intraday actual, over one week | `model_outputs.nc` |
| 12 | Short-term flexibility needs: annual totals (right) and distribution (left) | `model_outputs.nc` |
| 13 | Contributions to the short-term needs | `model_outputs.nc` |
| 14 | Ramping needs | `model_outputs.nc` |
| 15 | Contributions to the 25 largest ramping events | `model_outputs.nc` |
| 16 | Daily, weekly and seasonal flexibility needs | `model_outputs.nc` |
| 17 | Contributions to the daily, weekly and seasonal needs | `model_outputs.nc` |
| 18 | Net revenues of four assets from the day-ahead, intraday and reserve markets | day-ahead and intraday networks |

`model_outputs.nc` is `results/assessment/model_outputs.nc` in the run folder, which holds all flexibility indicators; see the User Manual, section 5, for every variable in it.

## Setup

### Import the necessary packages

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import pypsa
import xarray as xr
from datetime import datetime
from pathlib import Path

excel_data = {}


def export_data(sheet, data):
    excel_data[sheet] = pd.DataFrame(data).copy()

### Define folder and parameters

Define below the complete path to the folder of the run, and additional parameters of the run.

- `FOLDER` is the working directory of the run. The default, `../runs/teaser`, is where the README and the User Manual put the first run, seen from the `notebooks/` folder in which Jupyter opens this notebook.
- `ILLUSTRATIVE_WEEK` is the week plotted in Figure 11, counted from 0 for the first week of the year.
- With `EXPORT = 1` the notebook also saves the plotted data: one sheet per figure in an Excel file next to the notebook (see the last cell), and some of the data as CSV files in the run folder.

In [ ]:
# Change the line below to adjust for where your installation of the EU-Flex is
FOLDER = "../runs/teaser"

# Some graphs focus on a specific week of the year. Select below the week number
ILLUSTRATIVE_WEEK = 17

# Export the plotted data to Excel and CSV files or not
EXPORT = 0

### Import targeted PyPSA Network

Please specify the targeted network in the cell below to ensure the correct PyPSA network is imported/analysed. Note that by targeted network it is meant whether the day-ahead or intraday network is analysed and the CY and TY of this network. These are the networks of the final iteration, the same as the model outputs. The full list of networks for your run can be found in the `results/networks` folder of the run.

Only the day-ahead (DA) and intraday (ID) networks are solved and carry results. The day-ahead market is solved on the forecast timeseries of demand and renewable availability, the intraday market on the actual ones, starting from the day-ahead schedule. The networks under `resources/networks/` of the run (base, resampled, ...) hold only the inputs of the market models; the User Manual describes them in section 4. Unit names such as `N2 Nuclear` or `N3_Hydro` are those of the ACER Paper model; see `nda.generators` and `nda.storage_units` for the others.

The cell also opens `model_outputs.nc` and reads two hourly variables per bidding zone, in MW, that several figures use: the residual load (`residual_load`: demand minus available renewable generation, with the actual timeseries of the intraday market) and the short-term flexibility need (`short_term_need_hourly`: the difference between the actual residual load and its day-ahead forecast).

In [ ]:
DA_NETWORK = "day_ahead_network_2030_2013.nc"  # Specify the targeted network file for that specific run
ID_NETWORK = "intraday_network_2030_2013.nc"  # Specify the targeted network file for that specific run

# Load PyPSA network
nda = pypsa.Network(f"{FOLDER}/results/networks/{DA_NETWORK}")
nid = pypsa.Network(f"{FOLDER}/results/networks/{ID_NETWORK}")

results_base = xr.open_datatree(f"{FOLDER}/results/assessment/model_outputs.nc")

# Residual load (actual, as in the ID market) and short-term needs per bidding zone, in MW
res_load = results_base["base"]["residual_load"][0][0].T.to_pandas()
res_load.index = nda.snapshots
stneed = results_base["base"]["short_term_need_hourly"][0][0].T.to_pandas()
stneed.index = nda.snapshots

## Figure 9

Annual generation per carrier and bidding zone in the day-ahead market, in TWh, with the share of each carrier in the generation of the zone. The red markers show the annual demand of each zone.

In [ ]:
def plot_stacked_gen(n):
    gen_g = n.generators_t.p.sum()
    gen_g.name = "total_gen"
    gens = gen_g.index
    stu_g = n.storage_units_t.p.sum()
    stus = stu_g.index
    stu_g.name = "total_gen"

    gen_g = pd.concat(
        [gen_g, n.generators.bus[gens], n.generators.carrier[gens]], axis=1
    )
    stu_g = pd.concat(
        [stu_g, n.storage_units.bus[stus], n.storage_units.carrier[stus]], axis=1
    )
    tot_g = pd.concat([gen_g, stu_g])

    pivot = tot_g.pivot_table(index="carrier", columns="bus").total_gen / 1000000  # TWh

    idx = [i for i in pivot.index if "spillage" not in i]

    color = n.carriers.color[idx]

    plot_data = pivot.loc[idx].T.fillna(0)
    ax = plot_data.plot.bar(stacked=True, color=color, rot=0)

    totals = plot_data.sum(axis=1)

    for container, carrier in zip(ax.containers, plot_data.columns):
        labels = [
            f"{value / total:.0%}"
            if total != 0 and round(100 * value / total, 1) != 0
            else ""
            for value, total in zip(plot_data[carrier], totals)
        ]
        ax.bar_label(
            container,
            labels=labels,
            label_type="center",
            fontsize=7,
        )

    demand = n.loads_t.p.sum()
    demand.index = [i[0:2] for i in demand.index]
    demand_twh = demand.groupby(level=0).sum().reindex(plot_data.index) / 1_000_000
    demand_color = "#ff8080"

    ax.plot(
        range(len(plot_data)),
        demand_twh,
        linestyle="none",
        marker="_",
        markersize=16,
        markeredgewidth=3,
        color=demand_color,
        label="Demand",
    )

    for x, (total, load) in enumerate(zip(totals, demand_twh)):
        ax.annotate(
            f"{total:.0f} TWh",
            xy=(x, total),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontweight="bold",
        )
        ax.annotate(
            f"{load:.0f} TWh",
            xy=(x, load),
            xytext=(12, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            color=demand_color,
        )

    ax.set_ylim(0, max(totals.max(), demand_twh.max()) * 1.15)

    ax.legend(bbox_to_anchor=(1, 1), loc="upper left", fontsize="small")

    plt.ylabel("TWh")

    if EXPORT == 1:
        export_data("Figure 9", plot_data.assign(Total=totals, Demand=demand_twh))

    return ax


ax = plot_stacked_gen(nda)
ax.set_title("Total generation DA [TWh]")

## Figure 10

Annual renewable generation, curtailment and load spillage per bidding zone in the intraday market. The number in bold above each bar is the potential renewable generation of the zone (generation plus curtailment), and the percentage is the share of that potential that is curtailed. The total curtailment over all zones is printed in the plot.

In [ ]:
def plot_curtailment(n, unit="TWh"):
    renewables_id = [
        c for c in n.carriers.index if "Solar" in c or "Wind" in c or "renewables" in c
    ]

    # Get supply and curtailment to calculate potential renewable generation
    supply = n.statistics.supply(
        comps="Generator",
        groupby=["carrier", "bus"],
        nice_names=False,
    )

    curtailment = n.statistics.curtailment(
        comps="Generator",
        groupby=["carrier", "bus"],
        nice_names=False,
    )

    withdrawal = n.statistics.withdrawal(
        comps="Generator",
        groupby=["carrier", "bus"],
        nice_names=False,
    )

    # Potential supply = actual supply + curtailment
    potential_supply = supply + curtailment

    # Filter for renewable carriers that exist in the data
    existing_renewable_carriers = [
        carrier
        for carrier in renewables_id
        if carrier in potential_supply.index.get_level_values("carrier")
    ]

    if existing_renewable_carriers:
        renewable_potential = potential_supply.loc[existing_renewable_carriers]
        renewable_supply = supply.loc[existing_renewable_carriers]
        renewable_curtailment = curtailment.loc[existing_renewable_carriers]
    else:
        renewable_potential = potential_supply.iloc[0:0]
        renewable_supply = supply.iloc[0:0]
        renewable_curtailment = curtailment.iloc[0:0]

    # Group by bus and sum
    renewable_supply_by_bus = renewable_supply.groupby(level="bus").sum()
    renewable_potential_by_bus = renewable_potential.groupby(level="bus").sum()
    renewable_curtailment_by_bus = renewable_curtailment.groupby(level="bus").sum()
    load_spillage_by_bus = withdrawal.loc["load spillage"].groupby(level="bus").sum()

    curt = pd.DataFrame(
        [
            renewable_potential_by_bus,
            renewable_supply_by_bus,
            renewable_curtailment_by_bus,
            load_spillage_by_bus,
        ]
    ).T
    curt.columns = ["Potential", "Generation", "Curtailment", "Load spillage"]
    curt = curt.fillna(0)

    ### Plotting
    scaling = {"TWh": 1000000, "GWh": 1000, "MWh": 1}[unit]
    ax = (curt[["Generation", "Curtailment", "Load spillage"]] / scaling).plot.bar(
        stacked=True, rot=0
    )

    labels = [str(i) + "%" for i in (curt.Curtailment / curt.Potential * 100).round(1)]
    recs = ax.patches[0:3]
    for r, l in zip(recs, labels):
        h = r.get_height()
        ax.text(r.get_x() + r.get_width() / 2, h + 1, l, ha="center", va="bottom")

    plt.ylabel(unit)
    ct = curt.sum().Curtailment / curt.sum().Potential * 100
    ax.text(
        r.get_x(),
        60,
        "Total curtailment = {:.1f}%".format(ct),
        ha="center",
        va="bottom",
    )

    potential_twh = curt["Potential"] / 1_000_000
    bar_tops = (
        curt[["Generation", "Curtailment", "Load spillage"]].sum(axis=1) / scaling
    )

    for x, (top, potential) in enumerate(zip(bar_tops, potential_twh)):
        ax.annotate(
            f"{potential:.0f} TWh",
            xy=(x, top),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    ax.set_ylim(0, bar_tops.max() * 1.15)

    if EXPORT == 1:
        export_data(
            "Figure 10",
            (curt / scaling).assign(
                Curtailment_pct=curt.Curtailment / curt.Potential * 100
            ),
        )

    return ax, curt


ax, _ = plot_curtailment(nid)
ax.set_title("RES generation in ID")

## Figure 11

Residual load of all bidding zones together over the illustrative week: the day-ahead forecast against the intraday actual. The shaded difference is the forecast error that the intraday market has to absorb, which is the short-term flexibility need. The forecast is computed as the actual residual load minus the short-term need. Set `bus` to `"N1"`, `"N2"` or `"N3"` to plot a single zone.

In [ ]:
week, bus = ILLUSTRATIVE_WEEK, None  # None = all nodes; or "N1", "N2", "N3"
rl_id = res_load.sum(axis=1) if bus is None else res_load[bus]
rl_need = stneed.sum(axis=1) if bus is None else stneed[bus]
rl = (
    pd.DataFrame({"DA forecast": rl_id - rl_need, "ID actual": rl_id}).iloc[
        week * 168 : (week + 1) * 168
    ]
    / 1000
)

fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
ax.plot(
    rl.index,
    rl["DA forecast"],
    "--",
    color="tab:blue",
    linewidth=1.8,
    label="DA forecast",
)
ax.plot(rl.index, rl["ID actual"], color="tab:blue", linewidth=1.8, label="ID actual")
ax.fill_between(
    rl.index,
    rl["DA forecast"],
    rl["ID actual"],
    color="tab:blue",
    alpha=0.15,
    label="Forecast error",
)
ax.set(
    xlabel="Time",
    ylabel="Residual load [GW]",
    title=f"DA forecast versus ID actual — {bus or 'all nodes'}, week {week}",
)
ax.legend(ncol=3)
ax.grid(alpha=0.5)
plt.show()

if EXPORT == 1:
    export_data("Figure 11", rl)

## Figure 12 - Right

Annual short-term flexibility needs per bidding zone, split into upward needs (the actual residual load is above the forecast) and downward needs (it is below), from the variables `total_short_term_need_upward` and `total_short_term_need_downward`. Downward needs are negative.

In [ ]:
stneed_total = pd.DataFrame(
    {
        "Upwards": results_base["base"]["total_short_term_need_upward"][0][
            0
        ].to_pandas(),
        "Downwards": results_base["base"]["total_short_term_need_downward"][0][
            0
        ].to_pandas(),
    }
)
(stneed_total / 1000000).plot.bar(stacked=True, rot=0)
plt.title("Total short-term needs")
plt.ylabel("[TWh]")
plt.xlabel("")

if EXPORT == 1:
    export_data("Figure 12R", stneed_total / 1_000_000)

## Figure 12 - Left

Distribution of the hourly short-term needs (the residual load forecast errors) in each bidding zone, as a duration curve: the hours of the year sorted from the largest downward need (negative) to the largest upward need (positive).

In [ ]:
# Distribution of short-term errors
sorted_needs = pd.DataFrame(
    {bz: stneed[bz].sort_values(ignore_index=True) for bz in stneed}
)
sorted_needs.plot()
plt.title("Short-term needs\n(residual load forecast errors)")
plt.xlabel("Hours")
plt.ylabel("[MW]")

if EXPORT == 1:
    export_data("Figure 12L", sorted_needs)

## Figure 13

Contribution of each technology and of the interconnectors to covering the short-term needs of each bidding zone, in TWh, from the variables `short_term_technology_contributions_hourly` and `short_term_cross_border_contributions_hourly`. The contribution of renewable generation, labelled `RES`, is the part of the need that the other technologies do not cover. Load shedding and load spillage are left out.

In [ ]:
# Short-term contributions from the model outputs. The contribution of renewables
# (RES) is the part of the need that the other technologies do not cover.
stcont = results_base["base"]["short_term_technology_contributions_hourly"][0][0]
stcont = pd.DataFrame(
    stcont, columns=stcont.indexes["BZ"], index=stcont.indexes["carrier"]
).dropna()
stcont.loc["Interconnectors"] = results_base["base"][
    "short_term_cross_border_contributions_hourly"
][0][0]
stcont = stcont.drop(["Solar (PV)", "Wind Onshore"])
stcont.loc["RES"] = stneed.abs().sum() - stcont.sum()

# order for plot, leaving outside load shedding and spillage
idx = [
    "Interconnectors",
    "Nuclear",
    "Gas",
    "Hydro - Reservoir",
    "Batteries",
    "DSR_explicit",
    "DSR_implicit",
    "RES",
]

color = nda.carriers.color.copy()
color["Gas"] = "grey"
color["Interconnectors"] = "red"
color["RES"] = "green"

(stcont / 1000000).loc[idx].T.plot.bar(stacked=True, rot=0, color=color[idx])
plt.ylabel("TWh")
plt.title("Contributions to short-term needs")
plt.legend(bbox_to_anchor=(1, 1), loc="upper left", fontsize="small")


if EXPORT == 1:
    stcont.to_csv(f"{FOLDER}/shorterm_contributions.csv")
    export_data("Figure 13", (stcont / 1_000_000).loc[idx].T)

## Figure 14

Duration curve of the hourly ramps of the residual load (its change from one hour to the next, in MW/h) in each bidding zone. The ACER Paper configurations do not compute the ramping shortfall (`eva.ramping_shortfall.enable: false`), so it is not plotted here.

In [ ]:
# Plotting ramp
ramps = res_load.diff()
ramps_sorted = pd.concat(
    [ramps[c].sort_values(ignore_index=True) for c in ramps.columns], axis=1
)
ramps_sorted.plot()
plt.title("Ramping needs duration curve")
plt.xlabel("Hours")
plt.ylabel("[MW/h]")

if EXPORT == 1:
    export_data("Figure 14", ramps_sorted)

## Figure 15

Contribution of each technology and of the interconnectors to the 25 largest ramping events of each bidding zone, as a share of the ramp, from the variable `ramping_contribution_n25`. The part that the other technologies do not cover is labelled `Curtailment`: the contribution of renewable generation.

In [ ]:
ramp_cont25 = pd.DataFrame(
    results_base["base"]["ramping_contribution_n25"][0][0],
    columns=results_base["base"].indexes["carrier"],
    index=results_base["base"].indexes["BZ"],
).T

# Ramping contributions
ramp_cont = ramp_cont25.dropna().drop(["Solar (PV)", "Wind Onshore"])
ramp_cont.loc["Curtailment"] = 1 - ramp_cont.sum()
ramp_cont = ramp_cont.drop(["load spillage", "load shedding"])

idx = ramp_cont.index

color = nda.carriers.color.copy()
color.loc["Interconnectors"] = "green"
color.loc["Curtailment"] = "pink"

(ramp_cont * 100).T.plot.bar(stacked=True, rot=0, color=color[idx])
plt.legend(bbox_to_anchor=(1, 1), loc="upper left", fontsize="small")
plt.ylabel("[%]")
plt.title("Contributions to the 25 worst ramping events")


if EXPORT == 1:
    ramp_cont25.to_csv(f"{FOLDER}/ramping_contributions.csv")
    export_data("Figure 15", (ramp_cont * 100).T)

## Figure 16

Annual daily, weekly and seasonal flexibility needs, in TWh, stacked by bidding zone with the share of each zone, from the variables `daily_flexibility_need_aggregated`, `weekly_flexibility_need_aggregated` and `seasonal_flexibility_need`.

In [ ]:
fn_daily = results_base["base"]["daily_flexibility_need_aggregated"]
fn_weekly = results_base["base"]["weekly_flexibility_need_aggregated"]
fn_seasonal = results_base["base"]["seasonal_flexibility_need"]
flex_needs = pd.DataFrame(
    [fn_daily.values[0][0], fn_weekly.values[0][0], fn_seasonal.values[0][0]],
    index=["Daily", "Weekly", "Seasonal"],
    columns=nid.buses.index,
)
plot_data = (flex_needs / 1000000).fillna(0)
ax = plot_data.plot.bar(stacked=True, rot=0)
totals = plot_data.sum(axis=1)

# Percentage inside each node's segment
for container, node in zip(ax.containers, plot_data.columns):
    labels = [
        f"{value / total:.0%}" if total > 0 and value > 0 else ""
        for value, total in zip(plot_data[node], totals)
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=11)

# Total energy above each stacked bar
for x, total in enumerate(totals):
    ax.annotate(
        f"{total:.0f} TWh",
        xy=(x, total),
        xytext=(0, 6),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=11,
    )

ax.set_ylim(0, totals.max() * 1.15)
plt.legend(loc=4)
plt.title("Flexibility needs")
plt.ylabel("[TWh]")

if EXPORT == 1:
    flex_needs.to_csv(f"{FOLDER}/flex_needs.csv")
    export_data("Figure 16", plot_data)

## Figure 17

Share of each technology and of the interconnectors in covering the daily, weekly and seasonal needs of all bidding zones together, from the variables `technology_contributions_*` and `cross_border_contributions_*`. As in Figure 15, the part of the need that the other technologies do not cover is labelled `Curtailment`: the contribution of renewable generation. Load shedding and load spillage are left out.

In [ ]:
### Get contributions from Model Outputs

cont_daily = results_base["base"]["technology_contributions_daily"]
cont_weekly = results_base["base"]["technology_contributions_weekly"]
cont_seasonal = results_base["base"]["technology_contributions_seasonal"]
cont_xb_daily = results_base["base"]["cross_border_contributions_daily"]
cont_xb_weekly = results_base["base"]["cross_border_contributions_weekly"]
cont_xb_seasonal = results_base["base"]["cross_border_contributions_seasonal"]

conts = []
needs_names = ["Daily", "Weekly", "Seasonal"]
for i, (c, cxb) in enumerate(
    [
        (cont_daily, cont_xb_daily),
        (cont_weekly, cont_xb_weekly),
        (cont_seasonal, cont_xb_seasonal),
    ]
):
    df = pd.DataFrame(
        c[0][0].values, index=c.indexes["carrier"], columns=c.indexes["BZ"]
    )
    dfxb = pd.DataFrame(
        cxb.values[0], columns=cxb.indexes["BZ"], index=["Interconnectors"]
    )
    df = pd.concat([df, dfxb])
    df = df.drop(["Solar (PV)", "Wind Onshore"], axis=0).dropna()
    df.loc["Curtailment"] = flex_needs.loc[needs_names[i]] - df.sum()
    if needs_names[i] == "Daily":
        contribs_daily = pd.concat(
            [df, flex_needs.loc[["Daily"]].rename(index={"Daily": "Needs"})]
        )
    df.index.name = "carrier"
    df = df.reset_index()
    df = df.melt(id_vars=["carrier"], value_vars=c.indexes["BZ"], var_name="BZ")
    df["FlexNeed"] = needs_names[i]
    conts.append(df)

conts = pd.concat(conts)


# order for plot, leaving outside load shedding and spillage
idx = [
    "Interconnectors",
    "Nuclear",
    "Gas",
    "Hydro - Reservoir",
    "Batteries",
    "DSR_explicit",
    "DSR_implicit",
    "Curtailment",
]

contributions = conts.pivot_table(
    values="value", index="carrier", columns="FlexNeed", aggfunc="sum"
)
color["Curtailment"] = "pink"
color["Gas"] = "grey"
color = color[idx]
(contributions / contributions.sum() * 100).loc[idx, needs_names].T.plot.bar(
    stacked=True, rot=0, color=color
)
plt.ylabel("[%]")
plt.title("Contributions to flexibility needs, all nodes")
plt.legend(bbox_to_anchor=(1, 1), loc="upper left", fontsize="small")

if EXPORT == 1:
    contribs_daily.to_csv(
        f"{FOLDER}/contributions_flex_needs_daily_fromModelOutputsResLoad.csv"
    )
    contribs_daily.to_csv(
        f"{FOLDER}/contributions_flex_needs_daily_fromComputedResLoad.csv"
    )
    conts.to_csv(f"{FOLDER}/all_contributions.csv", index=False)
    export_data(
        "Figure 17", (contributions / contributions.sum() * 100).loc[idx, needs_names].T
    )

## Figure 18

Net revenues of four assets, as a share of their total, computed from the day-ahead and intraday networks:

- `DA`: the day-ahead energy margin, the revenue at the day-ahead price minus the variable costs and the start-up and shut-down costs;
- `ID`: the intraday margin on the adjustment to the day-ahead schedule, at the intraday price;
- `Reserves`: the revenues from the reserves (FCR and FRR) procured in the day-ahead market.

A negative share means the asset loses money in that market. Change `gs` to plot other assets; `cr_red` holds the net revenues of all generators and storage units, in k€/MW.

In [ ]:
def start_stop_costs(n, g):
    # Start-up and shut-down costs of a generator over the year
    switches = n.generators_t.status[g].diff(1)
    return (
        switches.clip(lower=0).sum() * n.generators.start_up_cost[g]
        - switches.clip(upper=0).sum() * n.generators.shut_down_cost[g]
    )


def net_revenues(nda, nid, g):
    # DA and ID energy margins and DA reserve revenues of a generator or storage unit
    if g in nda.generators.index:
        df, dfda_t, dfid_t = nda.generators, nda.generators_t, nid.generators_t
        suc_sdc_da = start_stop_costs(nda, g)
        suc_sdc_id = start_stop_costs(nid, g)
    elif g in nda.storage_units.index:
        df, dfda_t, dfid_t = nda.storage_units, nda.storage_units_t, nid.storage_units_t
        suc_sdc_da = suc_sdc_id = 0
    else:
        raise ValueError(f"{g} is not a generator or storage unit")
    b = df.bus[g]
    mc = df.marginal_cost[g]

    # DA: energy revenues minus variable and start-up costs
    p_da = dfda_t.p[g]
    margin_da = ((nda.buses_t.marginal_price[b] - mc) * p_da).sum() - suc_sdc_da

    # ID: the same for the adjustment to the DA schedule
    adj = dfid_t.p[g] - p_da
    margin_id = ((nid.buses_t.marginal_price[b] - mc) * adj).sum() - (
        suc_sdc_id - suc_sdc_da
    )

    # Reserve revenues
    reserves = []
    if df.reserve_FCR[g]:
        reserves += [i for i in nda.buses_t.keys() if ("FCR" in i) and ("price" in i)]
    if df.reserve_FRR[g]:
        reserves += [i for i in nda.buses_t.keys() if ("FRR" in i) and ("price" in i)]
    res_rev = sum(
        (nda.buses_t[res][b] * dfda_t[res.split("_price")[0]][g]).sum()
        for res in reserves
    )

    return pd.Series({"DA": margin_da, "ID": margin_id, "Reserves": res_rev})


# Net revenues per MW, in k€
assets = nda.generators.index.append(nda.storage_units.index)
p_nom = pd.concat([nda.generators.p_nom, nda.storage_units.p_nom])
cr_red = (
    pd.DataFrame({g: net_revenues(nda, nid, g) for g in assets}) / p_nom[assets] / 1000
)
gs = ["N1 Gas", "N1 BESS", "N2 Nuclear", "N3_Hydro"]

f, ax = plt.subplots(1)
(cr_red / cr_red.sum() * 100).T.loc[gs].plot.bar(stacked=True, rot=0, ax=ax)
plt.ylabel("[%]")
plt.legend(bbox_to_anchor=(1, 1), loc="upper left")
plt.title("Share of net revenues")

if EXPORT == 1:
    cr_red.to_csv(f"{FOLDER}/revenues.csv")
    export_data("Figure 18", (cr_red / cr_red.sum() * 100).T.loc[gs])

## Export

With `EXPORT = 1`, this cell writes the plotted data of every figure to an Excel file next to the notebook, with one sheet per figure, named after the notebook and the time of the export, and prints its path.

In [ ]:
if EXPORT == 1:
    excel_path = Path(f"teaser_case_analysis_{datetime.now():%Y%m%d_%H%M%S}.xlsx")

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for sheet, data in excel_data.items():
            data.to_excel(writer, sheet_name=sheet)

    print(excel_path.resolve())